# CareLoop Clinics — No-Show Guard
## Part 1 — Python & Data Foundations: ingest, validate, clean, enrich

This notebook covers Tasks 1.1–1.6. It reads the raw booking export, cleans it, adds two features,
enriches the clinic master with a live temperature reading, and writes the hand-off files used in Parts 2 and 3:

| File | Made in | Used in |
|---|---|---|
| `rejected_rows.json` | Task 1.1 | — |
| `clinics_enriched.csv` | Task 1.5 | Part 2 |
| `appointments_clean.csv` | Task 1.6 | Parts 2 and 3 |
| `load_data.sql` | Task 1.6 | Part 2 |

**Before running:** the first cell downloads the two original files (`appointments.csv`, `clinic.csv`)
from Google Drive. The originals are never edited; every cleaned file is a new file.

In [1]:
# Download the two original files (your own Google Drive copies)
!gdown 1CizURdkPBoVTQL5okia97IE0hP40c2TO
!gdown 1ya7MWlz5YAAGXL1iBVGez3e0r8_221Mi

Downloading...
From: https://drive.google.com/uc?id=1CizURdkPBoVTQL5okia97IE0hP40c2TO
To: /content/appointments.csv
100% 84.0k/84.0k [00:00<00:00, 4.86MB/s]


Downloading...
From: https://drive.google.com/uc?id=1ya7MWlz5YAAGXL1iBVGez3e0r8_221Mi
To: /content/clinic.csv
100% 491/491 [00:00<00:00, 1.51MB/s]


In [2]:
# Setup: imports and file locations
import csv
import json
import os

import numpy as np
import pandas as pd
import requests

APPOINTMENTS_PATH = "appointments.csv"   # original booking export (never edited)
CLINICS_PATH = "clinic.csv"              # original clinic master (never edited)

for path in [APPOINTMENTS_PATH, CLINICS_PATH]:
    print(f"{path}: {'found' if os.path.exists(path) else 'MISSING - run the download cell above'}")

appointments.csv: found
clinic.csv: found


## Task 1.1 — Read the file the foundations way

Using only the `csv` module: read every row with `csv.DictReader`, convert the numeric fields with
`parse_row`, and split the rows into **accepted** and **rejected** with `try/except ValueError`.
A row is rejected when one of its numeric fields cannot be converted (for example a blank `age`).
This is a **type** check only: an age of 250 converts to an integer, so that row is accepted here —
whether the value is *valid* is decided later, in Task 1.3.

In [3]:
with open(APPOINTMENTS_PATH, newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)

print("Header:", header)
print("Row count:", len(raw_rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020


In [ ]:
def parse_row(row):

    parsed = dict(row)
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed


accepted = []
rejected = []
for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as error:
        bad_row = dict(row)
        bad_row["error"] = str(error)
        rejected.append(bad_row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

# Why rows were rejected
blank_age = sum(1 for r in rejected if r["age"].strip() == "")
blank_distance = sum(1 for r in rejected if r["distance_km"].strip() == "")
print("Rejected with a blank age:", blank_age)
print("Rejected with a blank distance_km:", blank_distance)

# Impossible ages pass the type check (they are fixed in Task 1.3)
impossible_ages = [r["age"] for r in accepted if r["age"] < 1 or r["age"] > 100]
print("Accepted rows with an impossible age:", len(impossible_ages), impossible_ages)

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)
print("Saved rejected_rows.json")

Accepted rows: 988
Rejected rows: 32
Rejected with a blank age: 12
Rejected with a blank distance_km: 20
Accepted rows with an impossible age: 5 [0, 150, -4, 250, 199]
Saved rejected_rows.json


## Task 1.2 — Inspect and clean, pass 1

This starts again from the original `appointments.csv` with pandas (Task 1.1 was a separate exercise in
reading with the `csv` module). Steps, in order: (a) drop exact duplicate rows; (b) standardise
`booking_channel`; (c) convert the two date columns; (d) drop rows with no usable `appt_date` or no `status`.

In [5]:
df = pd.read_csv(APPOINTMENTS_PATH)

print("Shape:", df.shape)
print()
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Exact duplicate rows:", df.duplicated().sum())

Shape: (1020, 14)

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB

Missing values per column:
appointment_id      0
patient_id          0
clinic_id       

In [6]:
print("Rows at start:", len(df))

# (a) drop exact duplicate rows
df = df.drop_duplicates()
print("(a) after dropping exact duplicates:", len(df))

# (b) clean booking_channel spellings
print("    booking_channel distinct raw spellings:", df["booking_channel"].nunique())
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("    booking_channel distinct values after cleaning:", df["booking_channel"].nunique(),
      sorted(df["booking_channel"].dropna().unique()))
print("(b) after cleaning booking_channel:", len(df))

# (c) convert dates; unreadable appt_date values become NaT
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("    appt_date missing after conversion:", df["appt_date"].isna().sum())
print("    status missing:", df["status"].isna().sum())
print("(c) after converting dates:", len(df))

# (d) drop rows with no appt_date or no status
df = df.dropna(subset=["appt_date", "status"])
print("(d) after dropping missing appt_date / status:", len(df))

Rows at start: 1020
(a) after dropping exact duplicates: 1000
    booking_channel distinct raw spellings: 16
    booking_channel distinct values after cleaning: 3 ['App', 'Phone', 'Walk-In']
(b) after cleaning booking_channel: 1000
    appt_date missing after conversion: 6
    status missing: 4
(c) after converting dates: 1000
(d) after dropping missing appt_date / status: 990


## Task 1.3 — Clean, pass 2: validity, gaps and outliers

Order matters: invalid ages are set to missing first, then the gaps are filled, and only then are the
quartiles and fences computed, so that the fences describe the filled data.

In [7]:
# (a) ages below 1 or above 100 are impossible -> set to missing
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages set to missing:", invalid_age.sum(), sorted(df.loc[invalid_age, "age"].tolist()))
df.loc[invalid_age, "age"] = np.nan
print("Missing ages in total:", df["age"].isna().sum())

# (b) fill the gaps with the median
age_median = df["age"].median()
distance_median = df["distance_km"].median()
print("Median age:", age_median)
print("Median distance_km:", distance_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)
print("Missing after filling -> age:", df["age"].isna().sum(), "| distance_km:", df["distance_km"].isna().sum())

Invalid ages set to missing: 5 [-4.0, 0.0, 150.0, 199.0, 250.0]
Missing ages in total: 17
Median age: 48.0
Median distance_km: 8.25
Missing after filling -> age: 0 | distance_km: 0


In [8]:
# (c) cap outliers at the IQR fences
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_low = (df[col] < lower).sum()
    n_high = (df[col] > upper).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: Q1 = {q1}, Q3 = {q3}, IQR = {iqr}, lower fence = {lower}, upper fence = {upper}")
    print(f"    capped {n_low + n_high} values ({n_low} low, {n_high} high)")

# (d) final rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest fee after capping and rounding:", df["fee_inr"].max())
print("Largest distance after capping and rounding:", df["distance_km"].max())

distance_km: Q1 = 4.625, Q3 = 13.0, IQR = 8.375, lower fence = -7.9375, upper fence = 25.5625
    capped 26 values (0 low, 26 high)
fee_inr: Q1 = 500.0, Q3 = 787.5, IQR = 287.5, lower fence = 68.75, upper fence = 1218.75
    capped 3 values (0 low, 3 high)
Largest fee after capping and rounding: 1219
Largest distance after capping and rounding: 25.6


## Task 1.4 — Features and a NumPy summary

`lead_days` is the number of days between booking and appointment; `no_show` is the 0/1 target.
The summary below uses NumPy only, without loops.

In [9]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days runs from", df["lead_days"].min(), "to", df["lead_days"].max())
n_no_show = df["no_show"].sum()
print(f"No-shows: {n_no_show} of {len(df)} ({round(100 * n_no_show / len(df), 1)}%)")

lead_days runs from 1 to 45
No-shows: 218 of 990 (22.0%)


In [10]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) column means
col_means = X.mean(axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

# (b) bookings made more than 14 days ahead
mask = X[:, 3] > 14
print(f"Rows with lead_days > 14: {mask.sum()} ({round(100 * mask.mean(), 1)}%)")

# (c) z-scores of distance_km (np.std = population standard deviation)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
Rows with lead_days > 14: 296 (29.9%)
Rows with |z| > 2 for distance_km: 53


## Task 1.5 — Enrich the clinics from a live API

For each clinic, Open-Meteo returns an hourly temperature forecast. The first value of
`hourly.temperature_2m` (the midnight-UTC reading) is stored as `temp_c` — a stand-in, not the current
weather. Each clinic gets at most two attempts; if both fail, `temp_c` is left empty (never invented)
and the reason is printed.

In [11]:
clinics = pd.read_csv(CLINICS_PATH)
print("Clinic columns:", list(clinics.columns))
clinics

Clinic columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462


In [ ]:
URL = "https://api.open-meteo.com/v1/forecast"
name_col = "clinic_name" if "clinic_name" in clinics.columns else "name"


def fetch_temp(latitude, longitude):
    """Return (temp_c, reason). temp_c is None if the reading could not be obtained."""
    params = {"latitude": latitude, "longitude": longitude,
              "hourly": "temperature_2m", "forecast_days": 1}
    try:
        response = requests.get(URL, params=params, timeout=10)
    except requests.exceptions.RequestException as error:
        return None, f"request failed: {error}"
    print(f"    status code {response.status_code}")
    if response.status_code != 200:
        return None, f"status code {response.status_code}"
    data = response.json()
    if "hourly" not in data or not data["hourly"].get("temperature_2m"):
        return None, "no hourly temperature_2m in the reply"
    return data["hourly"]["temperature_2m"][0], None


temps = []
for _, clinic in clinics.iterrows():
    print(clinic[name_col])
    temp, reason = fetch_temp(clinic["latitude"], clinic["longitude"])
    if temp is None:
        print(f"    first attempt failed ({reason}) - retrying once")
        temp, reason = fetch_temp(clinic["latitude"], clinic["longitude"])
    if temp is None:
        print(f"    NO TEMPERATURE for {clinic[name_col]}: {reason} - temp_c left empty")
    else:
        print(f"    temp_c = {temp}")
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print()
print("Saved clinics_enriched.csv:", clinics.shape)
clinics

## Task 1.6 — Export the clean table and write the SQL load script

The clean table is sorted by `appointment_id`, dates are written as `YYYY-MM-DD`, and exactly 16 columns
are kept. `load_data.sql` holds one `INSERT` per clinic (first, because `appointments.clinic_id` is a
foreign key to `clinics`) and one per appointment: text in single quotes, numbers unquoted, and a missing
temperature written as an unquoted `NULL`.

*Building SQL text with f-strings is acceptable here only because this is a fixed, controlled dataset with
no apostrophes in any value; real applications use parameterised queries instead.*

In [13]:
COLUMNS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
           "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent", "prior_no_shows",
           "status", "lead_days", "no_show"]

clean = df.sort_values("appointment_id").copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")
clean = clean[COLUMNS]

# Checks before export
print("appointment_id unique:", clean["appointment_id"].is_unique,
      f"({clean['appointment_id'].nunique()} distinct values)")
missing_total = clean.isnull().sum().sum()
print("Missing values across the 16 columns:", missing_total)
assert clean["appointment_id"].is_unique, "appointment_id is not unique"
assert missing_total == 0, "there are missing values in the clean table"

clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", clean.shape)

appointment_id unique: True (990 distinct values)
Missing values across the 16 columns: 0
Saved appointments_clean.csv: (990, 16)


In [14]:
TEXT_COLUMNS = {"appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "gender",
                "booking_channel", "visit_type", "reminder_sent", "status"}


def sql_value(value, is_text):
    """Format one value for an INSERT statement."""
    if pd.isna(value):
        return "NULL"
    if is_text:
        return f"'{value}'"
    return f"{value}"


clinics_out = pd.read_csv("clinics_enriched.csv")
clinic_text_columns = {clinics_out.columns[0], clinics_out.columns[1], "city", "tier"}

lines = []
for _, row in clinics_out.iterrows():
    values = ", ".join(sql_value(row[col], col in clinic_text_columns) for col in clinics_out.columns)
    lines.append(f"INSERT INTO clinics VALUES ({values});")
n_clinic_inserts = len(lines)

for _, row in clean.iterrows():
    values = ", ".join(sql_value(row[col], col in TEXT_COLUMNS) for col in COLUMNS)
    lines.append(f"INSERT INTO appointments VALUES ({values});")
n_appointment_inserts = len(lines) - n_clinic_inserts

with open("load_data.sql", "w") as f:
    f.write("\n".join(lines) + "\n")

print("INSERT INTO clinics statements:", n_clinic_inserts)
print("INSERT INTO appointments statements:", n_appointment_inserts)
print("Total INSERT statements in load_data.sql:", len(lines))
print()
print("First lines of load_data.sql:")
for line in lines[:2] + lines[n_clinic_inserts:n_clinic_inserts + 2]:
    print(line)

INSERT INTO clinics statements: 8
INSERT INTO appointments statements: 990
Total INSERT statements in load_data.sql: 998

First lines of load_data.sql:
INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 17.9);
INSERT INTO clinics VALUES ('C02', 'Hadapsar Care Centre', 'Pune', 'Metro', 18.5089, 73.926, 18.5);
INSERT INTO appointments VALUES ('A0001', 'P509', 'C05', '2025-01-24', '2025-01-25', 60, 'F', 18.0, 850, 'Phone', 'Consultation', 'Yes', 1, 'No-Show', 1, 1);
INSERT INTO appointments VALUES ('A0002', 'P046', 'C07', '2025-05-27', '2025-06-06', 39, 'M', 12.5, 350, 'Walk-In', 'Follow-up', 'No', 2, 'Attended', 10, 0);


## Part 1 summary

- Exact duplicates and rows with no usable date or status were removed (row counts after each step are
  printed in Task 1.2).
- Invalid ages were treated as missing and filled with the median; distance and fee outliers were capped at the
  IQR fences (counts printed in Task 1.3).
- The overall no-show rate and the `lead_days` summary are printed in Task 1.4.
- Hand-off files written: `rejected_rows.json`, `clinics_enriched.csv`, `appointments_clean.csv`, `load_data.sql`.

## Part 2 check — Task 2.2 counts reproduced in Python

Same clinic counts and no-show counts as the SQL query in `02_queries.sql`.

In [15]:
appts = pd.read_csv("appointments_clean.csv")
clinics_check = pd.read_csv("clinics_enriched.csv")

by_clinic = (appts.groupby("clinic_id")
                  .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
                  .reset_index())
by_clinic["no_show_pct"] = (100 * by_clinic["no_shows"] / by_clinic["appointments"]).round(1)
by_clinic = (by_clinic.merge(clinics_check[["clinic_id", "clinic_name", "city"]], on="clinic_id")
                      [["clinic_name", "city", "appointments", "no_shows", "no_show_pct"]]
                      .sort_values("no_show_pct", ascending=False))
print(by_clinic.to_string(index=False))
print("Total appointments:", by_clinic["appointments"].sum(), "| total no-shows:", by_clinic["no_shows"].sum())

             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8
Total appointments: 990 | total no-shows: 218
